## Fetch the Github Repository Markdown Files

In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [2]:
# this my personal repository that I use for storing data engineering notes
learning_data_engineering_docs = read_repo_data('Joyan9', 'learning_data_engineering')
print(f"LDE documents: {len(learning_data_engineering_docs)}")

LDE documents: 47


## Create Text-based Search Index

In [3]:
from minsearch import Index

lde_index = Index(
    text_fields = ['content', 'filename'],
    keyword_fields=[]
)

lde_index.fit(learning_data_engineering_docs)

## Create Function to Log LLM Interactions for Evaluation

In [4]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [5]:
from pydantic_ai.messages import ModelMessagesTypeAdapter
import json, secrets
from pathlib import Path
from datetime import datetime

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

def log_entry(agent, messages, source="user"):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())
    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)
    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions[0],
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source,
    }

def log_interaction_to_file(agent, messages, source="user"):
    entry = log_entry(agent, messages, source)
    ts = entry["messages"][-1]["timestamp"]
    ts_str = ts.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)
    filename = f"{entry['agent_name']}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename
    with filepath.open("w", encoding="utf-8") as f:
        json.dump(entry, f, indent=2, default=serializer)
    return filepath

## Create Questions Generator Agent For Evaluation

In [8]:
from pydantic import BaseModel
from typing import List, Any
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

class QuestionsList(BaseModel):
    questions: list[str]

question_generation_prompt = """
You are helping create test questions for an AI study assistant.
The assistant has access to personal data engineering study notes.

Based on the provided note content, generate realistic questions a student 
might ask while studying. Questions should be natural and varied — 
some specific, some conceptual.

Generate one question per note provided.
""".strip()

question_generator = Agent(
    GroqModel('llama-3.3-70b-versatile'),
    system_prompt=question_generation_prompt,
    output_type=QuestionsList,
)

In [9]:
import random

sample = random.sample(learning_data_engineering_docs, 5)
prompt_docs = [d['content'][:500] for d in sample]  # truncate to save tokens
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions
print(questions)

['What is the difference between batch processing and stream processing in data engineering?', 'Can you explain the concept of a data contract and how it is used in data engineering?', 'What are the key components of the Spark driver, and how does it handle processing throughput?', 'How does Apache Flink handle bounded and unbounded data streams, and what are the roles of its components?', 'What are some best practices for designing data pipelines in Python, and how can you implement them in a real-world scenario?']


## AI Agents Answers Generated Questions and Logs Interactions

In [ ]:
import time

# Simplify the docstring to be very direct
def text_search(query: str) -> List[Any]:
    """Search the study notes for a specific topic.

    Args:
        query: The search term or question to look up.
    """
    return lde_index.search(query, num_results=5)

# Explicitly tell the model HOW to call tools in the system prompt
system_prompt = """
You are a helpful study assistant.
To answer questions, you MUST call the text_search tool with a JSON argument.
Example: {"query": "your search here"}
Provide clear, structured answers based on the study notes.
Always mention the source filename.
""".strip()

agent = Agent(
    name="lde_agent",
    instructions=system_prompt,
    tools=[text_search],
    model="groq:meta-llama/llama-4-scout-17b-16e-instruct"
)
 
for q in questions:
    result = await agent.run(q)
    log_interaction_to_file(agent, result.new_messages(), source="user")
    time.sleep(5)
    print(f"✓ {q}")

***The generated logs will be used for evaluation***